In [1]:
import pandas as pd
 
IN_PATH = "interest_rate.csv"
OUT_PATH = "interest_rate_yearly.csv"
START_YEAR, END_YEAR = 2015, 2024

In [3]:
# Load data set
df = pd.read_csv(IN_PATH)

In [5]:
# Clean data set
df = df.rename(columns={"observation_date": "date", "MORTGAGE30US": "rate"})
df["date"] = pd.to_datetime(df["date"], errors="coerce")
# FRED sometimes writes "." for missing values; coerce turns those into NaN
df["rate"] = pd.to_numeric(df["rate"], errors="coerce")
 
n_raw = len(df)
df = (
    df.dropna(subset=["date", "rate"])   # drop unparseable dates / missing rates
      .drop_duplicates(subset="date")    # one observation per week
      .sort_values("date")
      .reset_index(drop=True)
)
print(f"Rows: {n_raw} raw -> {len(df)} after cleaning")
# Sanity check: rates should be plausible percentages
assert df["rate"].between(0, 25).all(), "Rate outside 0-25% - check the data"

Rows: 521 raw -> 521 after cleaning


In [7]:
# Keep only the study window: 2015 - 2024
df["year"] = df["date"].dt.year
df = df[df["year"].between(START_YEAR, END_YEAR)]

In [9]:
# Aggregate weekly -> yearly
yearly = (
    df.groupby("year")["rate"]
      .agg(
          mortgage_rate_mean="mean",      # primary variable for the model
          mortgage_rate_median="median",
          mortgage_rate_min="min",
          mortgage_rate_max="max",
          n_weeks="count",
      )
      .reset_index()
)

# Year-end rate (last weekly observation of each year), as an alternative measure
yearly["mortgage_rate_yearend"] = df.groupby("year")["rate"].last().values
 
# Change in the annual average rate vs. the prior year (percentage points)
yearly["mortgage_rate_change"] = yearly["mortgage_rate_mean"].diff()
yearly = yearly.round(3)

In [11]:
# Validate
expected_years = set(range(START_YEAR, END_YEAR + 1))
assert set(yearly["year"]) == expected_years, "Missing years in output"
short = yearly[yearly["n_weeks"] < 50]
if not short.empty:
    print("WARNING: years with fewer than 50 weekly observations:\n", short[["year", "n_weeks"]])

In [13]:
# Save new file
yearly.to_csv(OUT_PATH, index=False)
print(yearly.to_string(index=False))
print(f"\nSaved to {OUT_PATH}")

 year  mortgage_rate_mean  mortgage_rate_median  mortgage_rate_min  mortgage_rate_max  n_weeks  mortgage_rate_yearend  mortgage_rate_change
 2015               3.851                 3.860               3.59               4.09       52                   4.01                   NaN
 2016               3.654                 3.590               3.41               4.32       52                   4.32                -0.197
 2017               3.990                 3.945               3.78               4.30       52                   3.99                 0.336
 2018               4.545                 4.550               3.95               4.94       52                   4.55                 0.555
 2019               3.936                 3.795               3.49               4.51       52                   3.74                -0.609
 2020               3.112                 3.070               2.66               3.72       53                   2.67                -0.824
 2021               